In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from model1 import run_torch_version
import json

def result_record(alg_name, ret, dataset, param=''):
    # 1. 动态构造键名 (Key)
    key = f"{dataset}_{alg_name}_{param}" if param else f"{dataset}_{alg_name}"
    
    # 2. 构造字典对象
    record_dict = {key: ret}
    
    # 3. 追加写入文件
    with open("result.jsonl", 'a') as f:
        # json.dumps 会自动给键加上双引号，并将元组 (0.6..., ...) 转换为列表 [0.6..., ...]
        f.write(json.dumps(record_dict) + '\n')

def modified_kmeans_fast_log_partitioned(mi_matrix, node_communities, tolerance=1e-7):
    # 1. 预处理：取对数拉伸低值区差异
    epsilon = 1e-9
    log_mi_matrix = np.log(np.clip(mi_matrix, epsilon, None))

    n = mi_matrix.shape[0]
    triu_indices = np.triu_indices(n, k=1)
    rows_all, cols_all = triu_indices
    all_log_values = log_mi_matrix[triu_indices]
    all_raw_values = mi_matrix[triu_indices]

    # 2. 分组逻辑
    same_mask = np.array([
        node_communities.get(r, -1) == node_communities.get(c, -2) 
        for r, c in zip(rows_all, cols_all)
    ])

    def find_active_cluster(log_v, raw_v, r_idx, c_idx):
        if len(log_v) == 0: return {}, {}
        
        # 在 Log 空间初始化中心点
        fixed_centroid = np.min(log_v) 
        centroid = np.max(log_v)
        
        is_stable = False
        while not is_stable:
            dist_to_fixed = np.abs(log_v - fixed_centroid)
            dist_to_active = np.abs(log_v - centroid)
            active_mask = dist_to_active < dist_to_fixed
            
            new_centroid = np.mean(log_v[active_mask]) if np.any(active_mask) else centroid
            if abs(new_centroid - centroid) < tolerance:
                is_stable = True
            centroid = new_centroid
            
        final_mask = (np.abs(log_v - centroid) < np.abs(log_v - fixed_centroid))
        
        # 构造两个簇的字典
        active_dict = dict(zip(zip(r_idx[final_mask], c_idx[final_mask]), raw_v[final_mask]))
        fixed_dict = dict(zip(zip(r_idx[~final_mask], c_idx[~final_mask]), raw_v[~final_mask]))
        return active_dict, fixed_dict

    # 3. 分类执行聚类
    cluster_same, fixed_same = find_active_cluster(
        all_log_values[same_mask], all_raw_values[same_mask], rows_all[same_mask], cols_all[same_mask]
    )
    cluster_diff, fixed_diff = find_active_cluster(
        all_log_values[~same_mask], all_raw_values[~same_mask], rows_all[~same_mask], cols_all[~same_mask]
    )

    # 4. 合并并返回两个字典 (解决解包报错)
    return {**cluster_same, **cluster_diff}, {**fixed_same, **fixed_diff}

def modified_kmeans_fast(mi_matrix, tolerance=1e-7):
    n = mi_matrix.shape[0]
    
    # 1. 提取上三角非零元素 (因为 mi_matrix 对称，且 j > i)
    # 这一步将 O(n^2) 的搜索范围缩小到实际有效的值
    triu_indices = np.triu_indices(n, k=1)
    all_values = mi_matrix[triu_indices]
    
    # 过滤掉 <= 0 的值 (对应原代码 if mi_matrix[i,j] <= 0: continue)
    valid_mask = all_values > 0
    values = all_values[valid_mask]
    # 记录这些有效值在原矩阵中的位置，最后还原字典用
    rows = triu_indices[0][valid_mask]
    cols = triu_indices[1][valid_mask]

    # 2. 初始化中心点
    fixed_centroid = 0.0
    centroid = np.max(values) if len(values) > 0 else 0.0
    
    is_stable = False
    
    while not is_stable:
        # 3. 向量化分类：计算每个点到两个中心点的距离
        # 原条件: (val - fixed_centroid) <= abs(val - centroid)
        dist_to_fixed = np.abs(values - fixed_centroid)
        dist_to_active = np.abs(values - centroid)
        
        # active_mask 为 True 表示该值应属于 cluster (动态簇)
        # 对应原代码 else 分支
        active_mask = dist_to_active < dist_to_fixed
        
        # 4. 更新动态中心点 (centroid)
        if np.any(active_mask):
            new_centroid = np.mean(values[active_mask])
        else:
            new_centroid = centroid
            
        # 5. 检查收敛条件：中心点偏移量小于阈值则稳定
        if abs(new_centroid - centroid) < tolerance:
            is_stable = True
        
        centroid = new_centroid

    # 6. 一次性还原为字典输出 (如果你的后续逻辑必须用字典)
    # 注意：如果 n 非常大，建议直接返回 mask 以节省内存
    fixed_mask = ~active_mask
    
    cluster = dict(zip(zip(rows[active_mask], cols[active_mask]), values[active_mask]))
    fixed_cluster = dict(zip(zip(rows[fixed_mask], cols[fixed_mask]), values[fixed_mask]))

    return cluster, fixed_cluster

def fast_mi_and_prob(x):
    # 假设 x 的形状是 (n_features, m_samples)
    n, m = x.shape
    
    # 1. 计算每个变量为 1 和 0 的概率
    # 使用 .reshape(-1) 确保它们是一维数组，方便后续计算
    count_1 = x.sum(axis=1).get() if hasattr(x, 'get') else x.sum(axis=1)
    count_1 = count_1.astype(float)
    count_0 = m - count_1
    
    p_i1 = count_1 / m
    p_i0 = count_0 / m

    # 2. 计算联合分布计数 (n x n)
    # count_11[i, j] 是 i=1 且 j=1 的样本数
    count_11 = x @ x.T
    
    # 3. 这里的 count_1 是一维的 (n,)，利用广播机制计算其他组合
    # count_1[:, None] 将其变为 (n, 1)
    count_1_col = count_1[:, np.newaxis]
    count_1_row = count_1[np.newaxis, :]
    
    count_10 = count_1_col - count_11
    count_01 = count_1_row - count_11
    count_00 = m - (count_11 + count_10 + count_01)

    # 4. 计算条件概率矩阵 p[i, j] = p(j=1 | i=1)
    # 注意：这里 i 是行，j 是列。原代码逻辑 p[i,j] = p_i1_j1 / p_i1
    p_matrix = count_11 / (count_1_col + 1e-12)

    # 5. 计算互信息 MI
    mi_matrix = np.zeros((n, n))
    
    # 组合列表：(联合概率, 边际概率1, 边际概率2)
    # p_i 和 p_j 均为形状为 (n,) 的一维数组
    pairs = [
        (count_11, p_i1, p_i1), # (1,1)
        (count_10, p_i1, p_i0), # (1,0)
        (count_01, p_i0, p_i1), # (0,1)
        (count_00, p_i0, p_i0)  # (0,0)
    ]

    for c_ij, p_i_vec, p_j_vec in pairs:
        p_ij = c_ij / m
        # 计算边际概率的乘积矩阵 P(i)*P(j)
        # np.outer(p_i_vec, p_j_vec) 会生成 (n, n) 矩阵
        p_i_p_j = np.outer(p_i_vec, p_j_vec)
        
        # 掩码计算：只有当联合概率和边际概率乘积均大于 0 时才计算
        mask = (p_ij > 1e-12) & (p_i_p_j > 1e-12)
        
        # MI 公式项
        mi_matrix[mask] += p_ij[mask] * np.log(p_ij[mask] / p_i_p_j[mask])

    return p_matrix, mi_matrix

def fast_imi_and_prob(x):
    # 假设 x 的形状是 (n_features, m_samples)
    if hasattr(x, 'get'): x = x.get() # 如果是 cupy 数组转为 numpy
    n, m = x.shape
    
    # 1. 计算每个变量为 1 和 0 的概率
    count_1 = x.sum(axis=1).astype(float)
    count_0 = m - count_1
    
    p_i1 = count_1 / m
    p_i0 = count_0 / m

    # 2. 计算联合分布计数 (n x n)
    count_11 = x @ x.T
    
    # 3. 利用广播机制计算其他组合
    count_1_col = count_1[:, np.newaxis]
    count_1_row = count_1[np.newaxis, :]
    
    count_10 = count_1_col - count_11
    count_01 = count_1_row - count_11
    count_00 = m - (count_11 + count_10 + count_01)

    # 4. 计算条件概率矩阵 p[i, j] = p(j=1 | i=1)
    p_matrix = count_11 / (count_1_col + 1e-12)

    # 5. 计算 IMI
    imi_matrix = np.zeros((n, n))
    
    # 定义四个分量的配置：(联合计数, 行边缘概率, 列边缘概率, 是否为负贡献)
    # 这里的贡献符号 sign 对应公式中的 + 或 -
    components = [
        (count_11, p_i1, p_i1, 1),  # MI(1,1) -> 正向
        (count_00, p_i0, p_i0, 1),  # MI(0,0) -> 正向
        (count_10, p_i1, p_i0, -1), # -|MI(1,0)| -> 负向
        (count_01, p_i0, p_i1, -1)  # -|MI(0,1)| -> 负向
    ]

    for c_ij, p_row_vec, p_col_vec, sign in components:
        p_ij = c_ij / m
        # 计算 P(Xi)*P(Xj) 矩阵
        p_i_p_j = np.outer(p_row_vec, p_col_vec)
        
        # 避免 log(0) 或 除以 0
        mask = (p_ij > 1e-12) & (p_i_p_j > 1e-12)
        
        # 计算单项 MI
        term = np.zeros((n, n))
        term[mask] = p_ij[mask] * np.log(p_ij[mask] / p_i_p_j[mask])
        
        # 根据 sign 累加到最终矩阵
        if sign == 1:
            imi_matrix += term
        else:
            # 公式要求减去绝对值: -|MI|
            imi_matrix -= np.abs(term)

    return p_matrix, imi_matrix

def IC(Networkx_Graph, Seed_Set, Probability):

    tree = nx.DiGraph()
    tree.add_node(Seed_Set[0])
    new_active, Ans = Seed_Set.tolist(), Seed_Set.tolist()
    while new_active:
        # Getting neighbour nodes of newly activate node
        (targets, edges) = Neighbour_finder(Networkx_Graph, Probability, new_active)
        # Calculating if any nodes of those neighbours can be activated, if yes add them to new_ones.

        new_active = []

        for (node, target) in edges:
            if np.random.uniform(0, 1) < Probability[node, target]:
                if target not in Ans: #success infected
                    tree.add_edge(node, target)
                    new_active.append(target)
                    Ans.append(target)
        # Checking which ones in new_ones are not in our Ans...only adding them to our Ans so that no duplicate in Ans.

    return Ans, tree


def Neighbour_finder(g, p, new_active):
    targets = []
    edges = []
    for node in new_active:
        node_neighbors = list(g.neighbors(node))
        targets += node_neighbors
        for target in node_neighbors:
            edges.append((node,target))

    return (targets, edges)

def generate_infections(A, num_sim = 100):

    N = A.shape[0]
    S = np.zeros([num_sim, N])
    nx_graph = nx.from_numpy_array(A)
    trees = []
    while len(trees) < num_sim:
        seed = np.random.choice(np.arange(0, N), size=1)
        cascade, tree = IC(Networkx_Graph=nx_graph, Seed_Set=seed, Probability=A)
        if len(tree.nodes) >= 3:
            S[len(trees), cascade] = 1
            trees.append(tree)
    average_paths = 0
    for tree in trees:
        average_paths += len(tree.nodes())

    print("average length of infections: ", average_paths / len(trees))
    return S

In [ ]:
from collections import Counter


def get_size_factor(comm_id):
    community_counts = Counter(node_communities.values())
    size = community_counts[comm_id]
    # 使用 log 缩放可以防止参数爆炸，+1 是为了防止 log(1)=0
    return np.log1p(size)

np.random.seed(2023)
N = 3000       # -N 1000-3000
AVG_K = 15     # -k 15 (average_degree)
MAX_K = 50     # -maxk 50 (max_degree)
MU = 0.1       # -mu 0.1 (mu)
MIN_C = 20     # -minc 20 (min_community)
MAX_C = 50     # -maxc 50 (max_community)

# AVG_K = 10     # 降低平均度
# MAX_K = 30     # 降低最大度
# MIN_C = 30     # 增加最小社区规模，确保能容纳度数较高的节点
# MAX_C = 60
# MU = 0.1       # -mu 0.1 (mu)

# 必须指定的幂律指数 (使用常用值)
TAU1 = 2.0     # 度分布幂律指数
TAU2 = 2.0     # 社区规模幂律指数

# 由于参数约束较严格，我们增加最大迭代次数以防 ExceededMaxIterations 错误
MAX_I = 100000

# --- 生成 LFR Benchmark 图 ---
try:
    G = nx.generators.community.LFR_benchmark_graph(
        n=N, 
        tau1=TAU1, 
        tau2=TAU2, 
        mu=MU, 
        average_degree=AVG_K, 
        max_degree=MAX_K,         # 指定最大度
        min_community=MIN_C, 
        max_community=MAX_C,      # 指定最大社区规模
        max_iters=MAX_I,          # 增加迭代次数
        seed=42
    )

    print(f"✅ LFR网络生成成功！")
    print(f"生成的LFR网络节点数: {G.number_of_nodes()}")
    print(f"生成的LFR网络边数: {G.number_of_edges()}")

    # 获取地面真值社区
    communities = {frozenset(G.nodes[v]['community']) for v in G}
    print(f"真实的社区数量: {len(communities)}")
    
except nx.ExceededMaxIterations as e:
    print(f"❌ 生成失败: {e}")
    print("请尝试进一步调整参数（例如增加MAX_I或略微放宽社区规模约束）。")

unique_comms = sorted(list(set(tuple(G.nodes[n]['community']) for n in G.nodes)))

# 2. 创建一个映射字典：{原始集合: 新的数字编号}
comm_to_id = {comm: i for i, comm in enumerate(unique_comms)}

# 3. 生成最终的 node_communities，其 value 全部为数字
node_communities = {n: comm_to_id[tuple(G.nodes[n]['community'])] for n in G.nodes}

A = nx.to_numpy_array(G)
P = np.zeros((N, N))
for u, v in G.edges():
    c_u = node_communities[u]
    c_v = node_communities[v]
    if c_u == c_v:
        weight = np.random.uniform(0.05, 0.1)
    else:
        # 社区之间：核心修改点
        # 基础跨社区概率
        base_inter_weight = np.random.uniform(0.005, 0.02)
        
        # 规模增强因子：大社区与大社区之间概率更高
        # 归一化因子（例如除以平均规模的 log 值）以保持权重在合理区间
        size_boost = get_size_factor(c_u) * get_size_factor(c_v)
        
        # 最终权重映射，确保不会超过 0.5（IC 模型通常不建议单边概率过高）
        weight = base_inter_weight * size_boost
    
    # 保证概率上限
    weight = min(weight, 0.4)
    P[u, v] = weight
    P[v, u] = weight
A = A * P
# 调用适配的生成函数
# 建议 num_sim 设大一些（如 500+）以获得更准的 Lift 估计
S = generate_infections(A, num_sim=100)

✅ LFR网络生成成功！
生成的LFR网络节点数: 2500
生成的LFR网络边数: 26146
真实的社区数量: 80
average length of infections:  1636.43


In [63]:
mi_matrix, p_matrix = fast_imi_and_prob(S.T)
cluster, fixed_cluster = modified_kmeans_fast_log_partitioned(mi_matrix, node_communities) #log 296
threshold = max(fixed_cluster.values())
prune_network = np.zeros([N, N])
prune_network[mi_matrix > threshold] = 1.0
prune_network[mi_matrix <= threshold] = 0.0

In [3]:
prune_network = np.ones([N, N])

In [64]:
G = nx.from_numpy_array(A)

In [5]:
#check edge

def check_pruned_edges(G, prune_network):
    """
    检查图 G 中的边有多少被 prune_network 过滤掉了
    """
    # 1. 获取 G 中所有的边
    original_edges = list(G.edges())
    total_g_edges = len(original_edges)
    
    missing_edges = []
    
    # 2. 遍历 G 的边，检查在矩阵中的对应位置是否为 0
    for u, v in original_edges:
        # 确保索引不越界
        if u < prune_network.shape[0] and v < prune_network.shape[1]:
            if prune_network[u, v] == 0:
                missing_edges.append((u, v))
        else:
            # 如果节点索引超出了矩阵范围，记录为异常
            print(f"Warning: Node index ({u}, {v}) out of prune_network bounds.")

    # 3. 计算统计数据
    num_missing = len(missing_edges)
    missing_ratio = (num_missing / total_g_edges) * 100 if total_g_edges > 0 else 0
    
    print("-" * 30)
    print(f"原始图 G 总边数: {total_g_edges}")
    print(f"被剪枝掉的边数 (不在 prune_network 中): {num_missing}")
    print(f"漏掉比例 (FN 潜在来源): {missing_ratio:.2f}%")
    print("-" * 30)
    
    return missing_edges

missing = check_pruned_edges(G, prune_network)

------------------------------
原始图 G 总边数: 10599
被剪枝掉的边数 (不在 prune_network 中): 27
漏掉比例 (FN 潜在来源): 0.25%
------------------------------


In [6]:
count = np.sum(prune_network == 1)
print(count)

988521


In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from torch.linalg import inv, slogdet
import numpy as np
import networkx as nx
from kneed import KneeLocator
import torch.nn.functional as F

class RegularizedInferenceIC(nn.Module):
    def __init__(self, N, Cascades, InstancePartition, gamma, prune_network):
        super(RegularizedInferenceIC, self).__init__()
        
        self.N = N
        self.gamma = gamma
        
        # 优化参数：网络边的概率对数 (对应 alpha)
        self.A_param = nn.Parameter(torch.zeros((N, N)))
        
        # 处理剪枝网络并注册为 buffer
        prune_network[prune_network == 0] = 1e-5
        self.register_buffer('prune_network_tensor', torch.from_numpy(prune_network).float())

        # =================================================================
        # 预计算 1: 级联状态矩阵 X (直接利用 GPU 矩阵乘法替代循环)
        # =================================================================
        # Cascades 形状为 (L, N)，即 L 个级联，N 个节点。X_li 表示级联 l 中节点 i 的状态
        X_np = np.array(Cascades)
        self.register_buffer('X', torch.tensor(X_np, dtype=torch.float32))
        
        #+++2
        co_occurrence = np.dot(X_np.T, X_np)
        # 归一化，避免数值过大
        co_occurrence = co_occurrence / (X_np.shape[0] + 1e-8)
        # 将共现频率映射到参数空间
        # 因为后面用 softplus，所以这里可以做个逆映射或简单的线性缩放
        # 我们希望共现高的边，初始 w 更大
        init_weight = torch.from_numpy(co_occurrence).float() * 0.5
        self.A_param = nn.Parameter(init_weight)

        # =================================================================
        # 预计算 2: 实例划分掩码矩阵 M (用于计算正则化 Omega)
        # =================================================================
        unique_insts = list(set(InstancePartition.values()) if isinstance(InstancePartition, dict) else set(InstancePartition))
        num_insts = len(unique_insts)
        M = torch.zeros(N, num_insts, dtype=torch.float32)
        
        for u in range(N):
            inst_id = InstancePartition[u] if isinstance(InstancePartition, dict) else InstancePartition[u]
            idx = unique_insts.index(inst_id)
            M[u, idx] = 1.0
            
        self.register_buffer('M_matrix', M)
        
        counts = M.sum(dim=0).unsqueeze(1)
        counts[counts == 0] = 1.0 
        self.register_buffer('M_counts', counts)

    def _get_prob_matrix(self):
        A_prob = torch.sigmoid(self.A_param)
        # 屏蔽对角线并应用剪枝网络
        A_prob = A_prob * (1.0 - torch.eye(self.N, device=A_prob.device))
        A_prob = A_prob * self.prune_network_tensor
        return A_prob
    
    #+++2
    def _get_weights(self):
        # =================================================================
        # 优化 2: 直接建模 w (Softplus)
        # softplus(x) = log(1 + exp(x))，保证 w > 0 且在大值区梯度不消失
        # =================================================================
        W = F.softplus(self.A_param)
        
        # 应用剪枝和对角线屏蔽
        W = W * (1.0 - torch.eye(self.N, device=W.device))
        W = W * self.prune_network_tensor
        return W

    def forward(self):
        #+++2
        # A_prob = self._get_prob_matrix()
        eps = 1e-8
        
        # =================================================================
        # 1. 负对数似然 (NLL) 高速张量计算 (对应公式推导)
        # =================================================================
        # w_ij = -log(1 - alpha_ij)
        #+++2
        W = self._get_weights()
        A_prob = -torch.expm1(-W)
        
        # 计算 y_i^l = \sum_j x_j^l w_ij
        # X 形状 (L, N)，W.T 形状 (N, N)，结果 Y 形状 (L, N)
        Y = torch.matmul(self.X, W.T)
        
        # 计算 Loss = sum [ (1 - X) * Y  -  X * log(1 - e^{-Y}) ]
        # 项 1: -(1 - x_i^l)y_i^l 的相反数
        term1 = (1.0 - self.X) * Y
        
        #+++3
        denominator = Y + eps
        log_prob_active = torch.log(-torch.expm1(-Y) + eps)
        term2 = self.X * log_prob_active
        # 项 2: x_i^l * log(1 - e^{-y_i^l}) 的相反数
        # term2 = self.X * torch.log(1.0 - torch.exp(-Y) + eps)
        
        #+++1
        negative_mask = ((1.0 - self.X) * Y > 0).float()
        # 对负样本进行随机下采样 (假设只保留 10% 的负样本惩罚)
        sampling_rate = 0.1
        random_mask = (torch.rand_like(Y) < sampling_rate).float()
        effective_negative_mask = negative_mask * random_mask
        term1_sampled = term1 * effective_negative_mask
        
        # 对所有级联和节点求和，并除以总数做归一化，防止 loss 爆掉
        NLL = torch.sum(term1_sampled - term2) / (self.N * self.N)
        
        # =================================================================
        # 2. 高速计算 Regularization (Omega)
        # =================================================================
        A_sum = torch.matmul(self.M_matrix.T, A_prob)
        A_mean = A_sum / self.M_counts 
        A_approx = torch.matmul(self.M_matrix, A_mean) 
        Omega = torch.sum((A_prob - A_approx)**2) / (self.N * self.N) 
        
        # 3. 返回最终的损失
        return NLL + self.gamma * Omega


def post_processing(estimated_A, beta=1.0):
    """
    beta: Recall 偏置系数。
    beta > 1.0 会让模型更厌恶漏报(FN)，从而降低阈值提高 Recall。
    """
    thresholds = np.linspace(start=1e-6, stop=0.5, num=10000)
    FP_FN_diff = np.zeros([len(thresholds)])

    for i, t in enumerate(thresholds):
        # 估计的漏报项 (本应是边但被阈值切掉了)
        predicted_FN = np.sum(estimated_A[estimated_A < t])
        # 估计的误报项 (本不该是边但被保留了)
        predicted_FP = np.sum(1.0 - estimated_A[estimated_A >= t])

        # 修改点：通过权重 beta 强迫模型降低阈值
        # 当 beta=2.0 时，1个 FN 的代价等于 2个 FP
        FP_FN_diff[i] = np.abs(predicted_FP - beta * predicted_FN)

    best_t = thresholds[np.argmin(FP_FN_diff)]
    
    IG_mat = np.zeros_like(estimated_A)
    IG_mat[estimated_A >= best_t] = 1
    IG = nx.from_numpy_array(IG_mat)

    return best_t, IG

def post_processing_kneed(estimated_A):
    # 1. 获取排序后的概率
    probs = np.sort(estimated_A.flatten())[::-1]
    probs = probs[probs > 1e-5]
    x = np.arange(len(probs))
    
    # 2. 调用 KneeLocator
    # curve='convex': 曲线是凸的
    # direction='decreasing': 曲线是递减的
    kneedle = KneeLocator(x, probs, S=1.0, curve='convex', direction='decreasing')
    
    # 3. 获取拐点对应的索引和阈值
    best_idx = kneedle.knee # 拐点的索引
    if best_idx is None:
        best_t = 0.05 # 备选保守阈值
    else:
        best_t = probs[best_idx]
        
    # 4. 绘图展示 (可选，调试时非常有用)
    # kneedle.plot_knee() 
    
    IG_mat = (estimated_A >= best_t).astype(float)
    return best_t, nx.from_numpy_array(IG_mat)

def calculate_F1(IG,RG):

    ig_edges = IG.edges
    rg_edges = RG.edges

    TP = 0.0
    FP = 0.0
    FN = 0.0

    for (i,j) in ig_edges:
        if (i,j) in rg_edges or (j,i) in rg_edges:
            TP += 1.0
        else:
            FP += 1.0

    for (i,j) in rg_edges:
        if (i,j) not in ig_edges and (j,i) not in ig_edges:
            FN += 1.0
    
    print(TP,FP,FN)

    P = TP / (TP+FP)
    R = TP / (TP+FN)

    return round(P,3),round(R,3),round(2*P*R / (P+R),3)

In [65]:
C = node_communities

    
l = set()
for node in C:
    l.add(C[node])
print(len(l))

dict_c = dict()
for i, item in enumerate(l):
    dict_c[item] = i
    
for node in C:
    C[node] = dict_c[C[node]]
    
gamma = 0.05

80


In [10]:
from sklearn.metrics import roc_auc_score, average_precision_score

def calculate_binary_auc(IG, G):
    """
    基于二值化的预测图 IG 和真实图 G 计算指标。
    注意：这里的 AUC 仅代表该特定阈值下的单点表现。
    """
    # 1. 转换为邻接矩阵
    # 确保节点顺序一致
    nodes = sorted(G.nodes())
    adj_predict = nx.to_numpy_array(IG, nodelist=nodes)
    adj_true = nx.to_numpy_array(G, nodelist=nodes)
    
    # 2. 提取上三角部分（忽略对角线，适用于无向图）
    iu = np.triu_indices(len(nodes), k=1)
    y_true = (adj_true[iu] > 0).astype(int)
    y_predict = adj_predict[iu].astype(int)
    
    # 3. 安全检查
    if len(np.unique(y_true)) < 2:
        return 0.5, 0.0
    
    # 4. 计算指标
    # 注意：此时 y_predict 是 0/1，roc_auc 相当于计算梯形的面积
    binary_roc_auc = roc_auc_score(y_true, y_predict)
    binary_pr_auc = average_precision_score(y_true, y_predict)
    
    print(f"Binary ROC-AUC: {binary_roc_auc:.4f}")
    print(f"Binary PR-AUC: {binary_pr_auc:.4f}")
    
    return round(binary_roc_auc,4), round(binary_pr_auc,4)

In [35]:
import numpy as np
import networkx as nx

def post_processing_with_community(estimated_A, node_communities, beta=1.0):
    N = estimated_A.shape[0]
    comm_labels = np.array([node_communities[i] for i in range(N)])
    same_comm_mask = (comm_labels[:, None] == comm_labels[None, :])
    np.fill_diagonal(same_comm_mask, False)
    diff_comm_mask = ~same_comm_mask
    np.fill_diagonal(diff_comm_mask, False)

    # --- 1. 数据驱动：计算社区先验优势 (Advantage Ratio) ---
    # 意思是：模型自己认为同社区的边比跨社区活跃多少倍？
    mean_inner = np.mean(estimated_A[same_comm_mask])
    mean_outer = np.mean(estimated_A[diff_comm_mask]) + 1e-9
    # 比如 mean_inner=0.08, mean_outer=0.02，那么 ratio 就是 4.0
    advantage_ratio = np.clip(mean_inner / mean_outer, 1.0, 5.0) 

    thresholds = np.linspace(1e-6, 0.5, 1000)

    # --- 2. 搜索同社区阈值 t_inner (带有偏爱) ---
    # 核心：给同社区的 beta 乘上 advantage_ratio
    # 漏掉一条同社区的边，现在的代价是正常情况的 advantage_ratio 倍
    beta_inner = beta * advantage_ratio
    diff_inner = np.zeros(len(thresholds))
    
    for i, t in enumerate(thresholds):
        pred_fn = np.sum(estimated_A[same_comm_mask & (estimated_A < t)])
        pred_fp = np.sum(1.0 - estimated_A[same_comm_mask & (estimated_A >= t)])
        diff_inner[i] = np.abs(pred_fp - beta_inner * pred_fn)
        
    t_inner = thresholds[np.argmin(diff_inner)]

    # --- 3. 搜索跨社区阈值 t_outer (严苛标准) ---
    # 跨社区保持正常的 beta，没有优待
    diff_outer = np.zeros(len(thresholds))
    for i, t in enumerate(thresholds):
        pred_fn = np.sum(estimated_A[diff_comm_mask & (estimated_A < t)])
        pred_fp = np.sum(1.0 - estimated_A[diff_comm_mask & (estimated_A >= t)])
        diff_outer[i] = np.abs(pred_fp - beta * pred_fn)
        
    t_outer = thresholds[np.argmin(diff_outer)]

    # 打印观察结果（可注释掉）
    print(f"Data-driven Advantage Ratio: {advantage_ratio:.2f}")
    print(f"Inner Threshold: {t_inner:.5f} | Outer Threshold: {t_outer:.5f}")

    # --- 4. 构造最终图 ---
    IG_mat = np.zeros_like(estimated_A)
    IG_mat[same_comm_mask & (estimated_A >= t_inner)] = 1
    IG_mat[diff_comm_mask & (estimated_A >= t_outer)] = 1
    
    IG = nx.from_numpy_array(IG_mat)

    # 保持 return 不变
    best_t = (t_inner + t_outer) / 2
    return best_t, IG

In [66]:
import copy
from tqdm import tqdm
iterations = 10000
lr = 0.001
patience = 300      # 连续 300 轮 Loss 不下降则早停
best_loss = float('inf')
counter = 0         # 早停计数器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training on device: {device}")
np.fill_diagonal(prune_network, 0.0)

# 1. 模型初始化
model = RegularizedInferenceIC(N=N, 
                                Cascades=S, 
                                InstancePartition=C, 
                                gamma=gamma,
                                prune_network=prune_network).to(device)

# 2. 优化器与调度器
optimizer = torch.optim.Adam(model.parameters(), lr=lr) 

# 用于保存表现最好的模型权重
best_model_wts = copy.deepcopy(model.state_dict())

model.train()
pbar = tqdm(range(iterations), desc="Optimizing")

for i in pbar:
    optimizer.zero_grad()
    loss = model() 
    loss.backward() 
    
    # 可选：梯度裁剪，防止 NLL 导致的梯度爆炸
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step() 
    
    curr_loss = loss.item()
    
    # --- Early Stopping 逻辑 ---
    # 如果当前 Loss 有进步 (设定一个微小的阈值 1e-6)
    if curr_loss < best_loss - 1e-6:
        best_loss = curr_loss
        best_model_wts = copy.deepcopy(model.state_dict()) # 记录最佳状态
        counter = 0 # 重置计数器
    else:
        counter += 1
    
    # 更新 tqdm 的后缀显示
    if i % 10 == 0:
        pbar.set_postfix({"Loss": f"{curr_loss:.4f}", "Best": f"{best_loss:.4f}", "Patience": f"{counter}/{patience}"})

    # 定期输出详细信息
    if (i+1) % 1000 == 0:
        tqdm.write(f"Iteration {i+1}, Loss: {curr_loss:.4f}, LR: {optimizer.param_groups[0]['lr']}")

    # 触发早停
    if counter >= patience:
        tqdm.write(f"Early stopping at iteration {i+1}. Recovering best weights...")
        break

# --- 训练结束，恢复最佳权重 ---
model.load_state_dict(best_model_wts)

# --- 评估与后处理 ---
model.eval()
with torch.no_grad():
    # 提取估计的概率矩阵
    A_star = model._get_prob_matrix().cpu().numpy()

# 再次确保对角线和剪枝约束
A_star = A_star * prune_network
A_star[A_star <= 1e-5] = 0.0

# 调用后处理逻辑
# 建议：如果 FN 依然很高，尝试传入 beta=1.5 或 2.0
best_t, IG = post_processing_with_community(A_star, C) 
result_record("mymodel", calculate_binary_auc(IG, G), "LFR", param=f'n{N}auc')

Training on device: cpu


Optimizing:   0%|          | 0/10000 [00:00<?, ?it/s]

Optimizing:  10%|█         | 1002/10000 [01:08<09:33, 15.70it/s, Loss=0.9888, Best=0.9597, Patience=1/300]

Iteration 1000, Loss: 0.9597, LR: 0.001


Optimizing:  20%|██        | 2002/10000 [02:10<08:46, 15.18it/s, Loss=0.5862, Best=0.5733, Patience=10/300]

...

Iteration 8000, Loss: 0.0558, LR: 0.001


Optimizing:  90%|█████████ | 9002/10000 [09:01<00:59, 16.87it/s, Loss=0.0425, Best=0.0416, Patience=53/300]

Iteration 9000, Loss: 0.0420, LR: 0.001


Optimizing: 100%|██████████| 10000/10000 [09:59<00:00, 16.69it/s, Loss=0.0329, Best=0.0317, Patience=32/300]


Iteration 10000, Loss: 0.0329, LR: 0.001
Data-driven Advantage Ratio: 1.07
Inner Threshold: 0.10160 | Outer Threshold: 0.10010
Binary ROC-AUC: 0.5208
Binary PR-AUC: 0.0087
